In [13]:
import pandas as pd

# Caminho para o seu arquivo de dados
caminho_arquivo = r"C:\Users\adriano\Downloads\tcc_karina\output.csv"

# --- Passo A.1: Aquisição e Preparação de Dados ---
# Carregando o arquivo CSV para um DataFrame do pandas.
# O parâmetro 'sep' indica que as colunas são separadas por ';'.
# 'encoding' é especificado para evitar erros de leitura com caracteres especiais.
try:
    df = pd.read_csv(caminho_arquivo, sep=';', encoding='latin1')
except Exception as e:
    print(f"Erro ao ler o arquivo com encoding 'latin1': {e}")
    print("Tentando com 'utf-8'...")
    df = pd.read_csv(caminho_arquivo, sep=';', encoding='utf-8')


# --- Inspeção Inicial ---
# Exibindo as 5 primeiras linhas do DataFrame para uma visualização inicial da estrutura.
print("--- Amostra dos Dados (Primeiras 5 Linhas) ---")
print(df.head())
print("\n" + "="*50 + "\n")

# Exibindo informações gerais sobre o DataFrame.
# Isso inclui o nome de cada coluna, a contagem de valores não nulos e o tipo de dado (Dtype).
# É fundamental para identificar problemas de tipo de dado e valores ausentes.
print("--- Informações Gerais do DataFrame (Colunas, Tipos, Nulos) ---")
df.info()



--- Amostra dos Dados (Primeiras 5 Linhas) ---
           Unnamed: 0  Unnamed: 1    DT_REFER        DENOM_CIA  CD_CVM  \
0  00.000.000/0001-91  2015-12-31  2015-12-31  BCO BRASIL S.A.    1023   
1                 NaN  2016-12-31  2016-12-31  BCO BRASIL S.A.    1023   
2                 NaN  2017-12-31  2017-12-31  BCO BRASIL S.A.    1023   
3                 NaN  2018-12-31  2018-12-31  BCO BRASIL S.A.    1023   
4                 NaN  2019-12-31  2019-12-31  BCO BRASIL S.A.    1023   

  RECUPERACAO 1::Ativo Total  2::Passivo Total  \
0       FALSO     1388864529        1388864529   
1       FALSO     1387215686        1387215686   
2       FALSO     1353075042        1353075042   
3       FALSO     1396507474        1396507474   
4       FALSO     1452266807        1452266807   

   6.01::Caixa Líquido Atividades Operacionais  \
0                                   28068706.0   
1                                    4385042.0   
2                                  -46925241.0   
3      

C:\Users\adriano\AppData\Local\Temp\ipykernel_19504\3480386482.py:11: DtypeWarning: Columns (6,12,21,27,29,31,32,44,47,48,53,118,216,217,235,236,239,243,244,248,249,252) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(caminho_arquivo, sep=';', encoding='latin1')


In [15]:
import pandas as pd

# Caminho para o seu arquivo de dados
caminho_arquivo = r"C:\Users\adriano\Downloads\tcc_karina\output.csv"

# --- Passo A.1: Aquisição e Preparação de Dados (Versão Definitiva) ---

# 1. Definimos um dicionário para instruir o pandas a tratar as colunas
#    problemáticas como texto ('str'). Isso garante que nenhum dado seja
#    perdido ou mal interpretado durante a leitura.
colunas_com_tipos_mistos = {
    6: str, 12: str, 21: str, 27: str, 29: str, 31: str, 32: str,
    44: str, 47: str, 48: str, 53: str, 118: str, 216: str, 217: str,
    235: str, 236: str, 239: str, 243: str, 244: str, 248: str,
    249: str, 252: str
}

# 2. Carregamos o CSV, passando o parâmetro 'dtype' com nossas instruções.
#    O 'DtypeWarning' será eliminado.
try:
    df = pd.read_csv(
        caminho_arquivo,
        sep=';',
        encoding='latin1',
        dtype=colunas_com_tipos_mistos  # <-- Instrução chave para resolver o aviso
    )
    print("✅ Arquivo CSV carregado com sucesso utilizando encoding 'latin1'.")
    
except Exception as e:
    print(f"Erro ao ler o arquivo com encoding 'latin1': {e}")
    print("Tentando com 'utf-8'...")
    df = pd.read_csv(
        caminho_arquivo,
        sep=';',
        encoding='utf-8',
        dtype=colunas_com_tipos_mistos
    )
    print("✅ Arquivo CSV carregado com sucesso utilizando encoding 'utf-8'.")

print("\n" + "="*50 + "\n")

# --- Inspeção Inicial ---
print("--- Informações Gerais do DataFrame (Pós-Carregamento) ---")
df.info(verbose=True)

✅ Arquivo CSV carregado com sucesso utilizando encoding 'latin1'.


--- Informações Gerais do DataFrame (Pós-Carregamento) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4065 entries, 0 to 4064
Data columns (total 303 columns):
 #    Column                                                                                               Dtype  
---   ------                                                                                               -----  
 0    Unnamed: 0                                                                                           object 
 1    Unnamed: 1                                                                                           object 
 2    DT_REFER                                                                                             object 
 3    DENOM_CIA                                                                                            object 
 4    CD_CVM                                                               

In [17]:
import pandas as pd
import numpy as np # Importamos numpy para usar np.nan

# Assumindo que 'df' é o DataFrame carregado no passo anterior.

# --- Passo A.2: Limpeza Inicial ---

print("--- Iniciando a Etapa A.2: Limpeza Inicial ---")

# 1. Padronização dos nomes das colunas
print("\nNomes das colunas originais (amostra):", df.columns.tolist()[:5])
df.columns = df.columns.str.lower().str.replace(' ', '_', regex=False)
print("Nomes das colunas padronizados (amostra):", df.columns.tolist()[:5])

# 2. Verificação e remoção de duplicatas
duplicatas = df.duplicated().sum()
print(f"\nNúmero de linhas duplicadas encontradas: {duplicatas}")
if duplicatas > 0:
    df.drop_duplicates(inplace=True)
    print("Linhas duplicadas foram removidas.")
    # Resetar o índice após a remoção é uma boa prática
    df.reset_index(drop=True, inplace=True)

# 3. Correção de Tipos de Dados (conversão controlada)
# Focaremos nas colunas mais críticas primeiramente.
# Se os nomes das colunas forem diferentes, ajuste-os conforme necessário.

# Convertendo a coluna de valor para numérica
if 'vl_conta' in df.columns:
    print("\nConvertendo a coluna 'vl_conta' para tipo numérico...")
    df['vl_conta'] = pd.to_numeric(df['vl_conta'], errors='coerce')
    print("Conversão de 'vl_conta' concluída.")

# Convertendo a coluna de data para datetime
if 'dt_refer' in df.columns:
    print("\nConvertendo a coluna 'dt_refer' para tipo datetime...")
    df['dt_refer'] = pd.to_datetime(df['dt_refer'], errors='coerce')
    print("Conversão de 'dt_refer' concluída.")

print("\n--- Concluída a Etapa A.2: Limpeza Inicial ---")
print("\n" + "="*50 + "\n")


# --- Passo A.3: Análise de Dados Ausentes ---

print("--- Iniciando a Etapa A.3: Análise de Dados Ausentes ---")

# Contando o total de valores nulos por coluna
missing_values = df.isnull().sum()

# Filtrando para exibir apenas as colunas que de fato possuem valores nulos
missing_values_com_nulos = missing_values[missing_values > 0].sort_values(ascending=False)

if not missing_values_com_nulos.empty:
    print("\nColunas com valores ausentes e suas respectivas contagens:")
    # Calculando a porcentagem de valores ausentes
    percent_missing = (missing_values_com_nulos / len(df)) * 100
    missing_data_report = pd.DataFrame({
        'Total de Nulos': missing_values_com_nulos,
        'Porcentagem (%)': percent_missing.round(2)
    })
    print(missing_data_report)
else:
    print("\n✅ Excelente! Não foram encontrados valores ausentes no DataFrame.")

# Exibindo as informações do DataFrame para verificar as mudanças nos tipos de dados (Dtypes)
print("\n--- Informações do DataFrame Após Limpeza e Análise de Nulos ---")
df.info(verbose=True)

--- Iniciando a Etapa A.2: Limpeza Inicial ---

Nomes das colunas originais (amostra): ['Unnamed: 0', 'Unnamed: 1', 'DT_REFER', 'DENOM_CIA', 'CD_CVM']
Nomes das colunas padronizados (amostra): ['unnamed:_0', 'unnamed:_1', 'dt_refer', 'denom_cia', 'cd_cvm']

Número de linhas duplicadas encontradas: 0

Convertendo a coluna 'dt_refer' para tipo datetime...
Conversão de 'dt_refer' concluída.

--- Concluída a Etapa A.2: Limpeza Inicial ---


--- Iniciando a Etapa A.3: Análise de Dados Ausentes ---

Colunas com valores ausentes e suas respectivas contagens:
                                                    Total de Nulos  \
unnamed:_0                                                    3452   
1.02.01.08.03::créditos_com_controladores                     3102   
1.02.01.03::contas_a_receber                                  3102   
1.02.01.04::estoques                                          3102   
1.02.01.06::tributos_diferidos                                3102   
...                   

In [21]:
import pandas as pd
import numpy as np

# Assumindo que 'df' é o DataFrame resultante do passo anterior.

# --- Etapa A.3: Tratamento de Dados Ausentes (Versão Corrigida) ---

print("--- Executando Tratamento de Dados Ausentes (Remoção e Imputação) ---")

# 1. Definir o limiar para remoção de colunas (60%)
limiar_percentual = 60.0
percent_missing = (df.isnull().sum() / len(df)) * 100
colunas_para_remover = percent_missing[percent_missing > limiar_percentual].index.tolist()

if colunas_para_remover:
    print(f"\nLimiar definido: {limiar_percentual}%")
    print(f"As seguintes colunas serão removidas por exceder o limiar de valores nulos:")
    for col in colunas_para_remover:
        print(f"  - '{col}' ({percent_missing[col]:.2f}% nulos)")
    
    df.drop(columns=colunas_para_remover, inplace=True)
    print("\n✅ Colunas removidas com sucesso.")
else:
    print(f"\nNenhuma coluna excedeu o limiar de {limiar_percentual}% de valores nulos.")


# 2. Imputação de valores nulos para as colunas remanescentes

print("\n--- Iniciando processo de imputação ---")

colunas_numericas_com_nulos = df.select_dtypes(include=np.number).columns[
    df.select_dtypes(include=np.number).isnull().any()
].tolist()

if colunas_numericas_com_nulos:
    print("Imputando valores nulos em colunas numéricas com a MEDIANA:")
    for col in colunas_numericas_com_nulos:
        mediana = df[col].median()
        # --- CÓDIGO CORRIGIDO ---
        # Substituímos o uso de inplace=True pela reatribuição direta.
        # Isso elimina o FutureWarning e é a prática recomendada.
        df[col] = df[col].fillna(mediana)
        print(f"  - Nulos da coluna '{col}' preenchidos com o valor da mediana: {mediana:.2f}")
    print("\n✅ Imputação de colunas numéricas concluída.")
else:
    print("\nNenhuma coluna numérica com valores nulos para imputar.")


# 3. Verificação Final
print("\n--- Verificação Final de Valores Ausentes ---")
nulos_restantes = df.isnull().sum()
nulos_restantes = nulos_restantes[nulos_restantes > 0]

if nulos_restantes.empty:
    print("\n✅ Excelente! Todos os valores ausentes foram tratados com sucesso.")
else:
    print("\nAinda restam valores ausentes nas seguintes colunas:")
    print(nulos_restantes.sort_values(ascending=False))

--- Executando Tratamento de Dados Ausentes (Remoção e Imputação) ---

Nenhuma coluna excedeu o limiar de 60.0% de valores nulos.

--- Iniciando processo de imputação ---

Nenhuma coluna numérica com valores nulos para imputar.

--- Verificação Final de Valores Ausentes ---

Ainda restam valores ausentes nas seguintes colunas:
3.99.02.01::on                                         1187
3.99.01.01::on                                          459
3.99::lucro_por_ação_-_(reais_/_ação)                   227
7.06.03::outros                                         221
3.04.04::outras_receitas_operacionais                   221
7.01.01::vendas_de_mercadorias,_produtos_e_serviços     221
7.01.02::outras_receitas                                221
3.04.05::outras_despesas_operacionais                   221
7.02::insumos_adquiridos_de_terceiros                   221
7.02.04::outros                                         221
7.03::valor_adicionado_bruto                            221
7.05::valor

In [25]:
import pandas as pd

# Assumindo que 'df' é o DataFrame limpo e tratado da etapa anterior.

# --- Etapa A.4: Engenharia de Atributos (Parte 1 - Pivotagem, Versão Robusta) ---

print("--- Iniciando a Etapa A.4: Engenharia de Atributos ---")

# --- PASSO DE DIAGNÓSTICO ---
# Vamos inspecionar os nomes das colunas para encontrar o nome correto da coluna de valores.
print("\n[Diagnóstico] Lista completa de colunas no DataFrame antes da pivotagem:")
print(df.columns.tolist())
print("-" * 60)

# Verifique a lista impressa acima. O nome da coluna com os valores monetários
# (provavelmente algo como 'vl_conta', 'valor_conta', ou similar) deve ser usado abaixo.
# Por padrão, vamos manter 'vl_conta'. Se o nome for outro, ajuste na linha 'values='.
NOME_COLUNA_VALORES = 'vl_conta' 

# Verificação para garantir que a coluna existe antes de tentar a pivotagem
if NOME_COLUNA_VALORES not in df.columns:
    print(f"❌ ERRO CRÍTICO: A coluna '{NOME_COLUNA_VALORES}' não foi encontrada no DataFrame.")
    print("Por favor, verifique a lista de colunas acima e ajuste a variável 'NOME_COLUNA_VALORES' no código.")
else:
    print(f"\n✅ A coluna de valores '{NOME_COLUNA_VALORES}' foi encontrada. Prosseguindo com a pivotagem.")
    
    try:
        # A função pivot_table é robusta para lidar com possíveis duplicatas.
        df_pivot = pd.pivot_table(
            df,
            index=['cnpj', 'denom_cia', 'dt_refer', 'ordem_exerc'],
            columns='ds_conta',
            values=NOME_COLUNA_VALORES, # Usando a variável que definimos
            aggfunc='sum',
            fill_value=0
        )

        df_pivot.reset_index(inplace=True)
        # Limpeza de caracteres especiais que podem surgir nos nomes das colunas após pivotar
        df_pivot.columns = df_pivot.columns.str.replace('[^A-Za-z0-9_]+', '', regex=True)

        print("\n✅ DataFrame pivotado com sucesso.")
        print(f"Nova estrutura do DataFrame: {df_pivot.shape[0]} linhas e {df_pivot.shape[1]} colunas.")

        print("\n--- Amostra do DataFrame Pivotado ---")
        print(df_pivot.head())

    except Exception as e:
        print(f"\nOcorreu um erro inesperado durante a pivotagem: {e}")

--- Iniciando a Etapa A.4: Engenharia de Atributos ---

[Diagnóstico] Lista completa de colunas no DataFrame antes da pivotagem:
['unnamed:_1', 'dt_refer', 'denom_cia', 'cd_cvm', 'recuperacao', '1::ativo_total', '2::passivo_total', '6.01::caixa_líquido_atividades_operacionais', '6.01.01::caixa_gerado_nas_operações', '6.01.02::variações_nos_ativos_e_passivos', '6.01.03::outros', '6.02::caixa_líquido_atividades_de_investimento', '6.03::caixa_líquido_atividades_de_financiamento', '6.04::variação_cambial_s/_caixa_e_equivalentes', '6.05::aumento_(redução)_de_caixa_e_equivalentes', '6.05.01::saldo_inicial_de_caixa_e_equivalentes', '6.05.02::saldo_final_de_caixa_e_equivalentes', '5.01::saldos_iniciais', '5.02::ajustes_de_exercícios_anteriores', '5.03::saldos_iniciais_ajustados', '5.04::transações_de_capital_com_os_sócios', '5.04.01::aumentos_de_capital', '5.04.02::gastos_com_emissão_de_ações', '5.04.03::opções_outorgadas_reconhecidas', '5.04.04::ações_em_tesouraria_adquiridas', '5.04.05::açõe